# SE-ResNet50 Training - V2
**Improvements:**
- Proper validation split (15% from training data)
- Multi-metric evaluation (Acc, F1, Precision, Recall)
- Live overfitting monitoring every 5 epochs
- Composite scoring for best model selection
- Multiple checkpoint saving

**Model:** SE-ResNet50 with Squeeze-and-Excitation blocks  
**SE Reduction Ratio:** 16  
**Dataset:** Kermany OCT2017 

In [1]:
# IMPORTS
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from pathlib import Path
import numpy as np
import time
from tqdm import tqdm
from collections import defaultdict, Counter
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


In [2]:
# HELPER FUNCTIONS - WITH ALL v2 FIXES

def get_next_serial_number(checkpoint_dir):
    """Automatically detect the next available serial number."""
    import re
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        return 1
    
    existing = list(checkpoint_dir.glob("*.pth"))
    if not existing:
        return 1
    
    serial_numbers = []
    for f in existing:
        match = re.match(r'^(\d+)_', f.name)
        if match:
            serial_numbers.append(int(match.group(1)))
    
    return max(serial_numbers) + 1 if serial_numbers else 1


def save_checkpoint(model, optimizer, epoch, metrics, is_best, checkpoint_dir,
                   serial_number, model_name, seed, mode='intermediate'):
    """Save checkpoint with comprehensive metrics."""
    from datetime import datetime
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    serial_str = f"{serial_number:02d}"
    
    filename = f"{serial_str}_{model_name}_seed{seed}_epoch{epoch}_{mode}_{timestamp}.pth"
    filepath = checkpoint_dir / filename
    
    checkpoint = {
        'serial_number': serial_number,
        'model_name': model_name,
        'seed': seed,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'is_best': is_best,
        'mode': mode,
        'timestamp': timestamp
    }
    
    torch.save(checkpoint, filepath)
    print(f"Saved {mode}: {filename}")
    return filepath


def create_stratified_split(dataset, val_ratio=0.15, seed=42):
    """
    Create stratified train/val split maintaining class balance.
    
    ✅ FIX #1: Uses dataset.targets (fast) instead of loading images.
    Time savings: ~30 minutes → ~1 second for 76k images!
    """
    # ✅ FAST: ImageFolder already has labels loaded in .targets
    labels = np.array(dataset.targets)
    indices = np.arange(len(labels))
    
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_ratio,
        stratify=labels,
        random_state=seed
    )
    
    return train_idx, val_idx


def is_better_model(new_score, new_loss, new_acc, new_epoch,
                    best_score, best_loss, best_acc, best_epoch,
                    eps=1e-9):
    """
    Deterministic tie-breaking for model selection.
    
    ✅ Handles ties with clear priority rules.
    
    Priority:
    1. Higher composite score (primary)
    2. If tied: Lower validation loss
    3. If tied: Higher validation accuracy  
    4. If tied: Later epoch (more stable)
    
    Returns True if new model is better.
    """
    # Primary criterion: composite score
    if new_score > best_score + eps:
        return True
    
    if abs(new_score - best_score) <= eps:  # Scores are tied
        # Tie-break 1: Lower validation loss
        if new_loss < best_loss - eps:
            return True
        
        if abs(new_loss - best_loss) <= eps:  # Loss also tied
            # Tie-break 2: Higher validation accuracy
            if new_acc > best_acc + eps:
                return True
            
            if abs(new_acc - best_acc) <= eps:  # Acc also tied
                # Tie-break 3: Prefer later epoch (more stable)
                if new_epoch > best_epoch:
                    return True
    
    return False


def check_overfitting(train_acc, val_acc, train_loss, val_loss, threshold_acc=10.0, threshold_loss=0.5):
    """Check for overfitting based on train-val gaps."""
    acc_gap = train_acc - val_acc
    loss_gap = val_loss - train_loss
    
    is_overfitting = (acc_gap > threshold_acc) or (loss_gap > threshold_loss)
    
    return {
        'is_overfitting': is_overfitting,
        'acc_gap': acc_gap,
        'loss_gap': loss_gap,
        'severity': 'HIGH' if (acc_gap > 15.0 or loss_gap > 1.0) else 'MODERATE' if is_overfitting else 'NONE'
    }


print("Helper functions loaded (v2 with all fixes)")

Helper functions loaded (v2 with all fixes)


In [3]:
# CONFIGURATION

ROOT = Path(r"C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training")

CHECKPOINT_DIR = ROOT / "Checkpoints"
DATASET_ROOT = ROOT / "Data_Kermany_OCT2017"
TRAIN_PATH = DATASET_ROOT / "train"  # Will split this into train/val
TEST_PATH = DATASET_ROOT / "test"  # Reserved for final evaluation

# Model parameters
MODEL_NAME = "se_resnet"
NUM_EPOCHS = 50
SEED = 84
SE_REDUCTION = 16  # Squeeze-and-Excitation reduction ratio

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

# NEW: Validation split
VAL_SPLIT_RATIO = 0.15  # 15% of training data for validation

# NEW: Checkpointing strategy
SAVE_BEST_ONLY = False  # Save multiple checkpoints
SAVE_EVERY_N_EPOCHS = 5  # Save every 5 epochs

# NEW: Monitoring
OVERFITTING_CHECK_INTERVAL = 5  # Check every 5 epochs

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SERIAL_NUMBER = get_next_serial_number(CHECKPOINT_DIR)

print("="*80)
print("IMPROVED TRAINING CONFIGURATION - SE-RESNET50")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"SE Reduction: {SE_REDUCTION}")
print(f"Device: {DEVICE}")
print(f"\nValidation: {VAL_SPLIT_RATIO*100:.0f}% of training data (stratified)")
print(f"Checkpointing: Every {SAVE_EVERY_N_EPOCHS} epochs + best model")
print(f"Overfitting checks: Every {OVERFITTING_CHECK_INTERVAL} epochs")
print("="*80)

IMPROVED TRAINING CONFIGURATION - SE-RESNET50
Model: se_resnet
Serial: 05 | Seed: 84 | Epochs: 50
SE Reduction: 16
Device: cuda

Validation: 15% of training data (stratified)
Checkpointing: Every 5 epochs + best model
Overfitting checks: Every 5 epochs


In [4]:
# DATASET LOADING WITH IMPROVED VAL SPLIT

print("\n" + "="*80)
print("CREATING STRATIFIED TRAIN/VAL SPLIT")
print("="*80)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load full training dataset
full_dataset = ImageFolder(root=str(TRAIN_PATH))

print(f"\nOriginal training folder: {len(full_dataset):,} images")

# Create stratified split
train_idx, val_idx = create_stratified_split(full_dataset, VAL_SPLIT_RATIO, SEED)

print(f"\nStratified split created:")
print(f"  Training: {len(train_idx):,} images ({(1-VAL_SPLIT_RATIO)*100:.1f}%)")
print(f"  Validation: {len(val_idx):,} images ({VAL_SPLIT_RATIO*100:.1f}%)")

# Verify class balance
train_labels = [full_dataset[i][1] for i in train_idx]
val_labels = [full_dataset[i][1] for i in val_idx]

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

print("\nClass distribution:")
print(f"{'Class':<12} {'Training':>10} {'Validation':>12} {'Val %':>8}")
print("-" * 50)
for i, class_name in enumerate(CLASS_NAMES):
    train_count = train_counts[i]
    val_count = val_counts[i]
    val_pct = (val_count / (train_count + val_count)) * 100
    print(f"{class_name:<12} {train_count:>10,} {val_count:>12,} {val_pct:>7.1f}%")

# Create datasets with transforms
train_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=train_transform)
val_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=val_test_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(val_dataset_full, val_idx)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print("\n✓ Much larger validation set = more reliable model selection!")
print("="*80)


CREATING STRATIFIED TRAIN/VAL SPLIT

Original training folder: 55,792 images

Stratified split created:
  Training: 47,423 images (85.0%)
  Validation: 8,369 images (15.0%)

Class distribution:
Class          Training   Validation    Val %
--------------------------------------------------
CNV              19,006        3,354    15.0%
DME               5,862        1,034    15.0%
DRUSEN            3,280          579    15.0%
NORMAL           19,275        3,402    15.0%

DataLoaders created:
  Train batches: 1482
  Val batches: 262

✓ Much larger validation set = more reliable model selection!


In [5]:
# DATA OVERLAP VERIFICATION 
# Run this BEFORE training to verify dataset cleanliness

print("="*80)
print("VERIFYING NO TRAIN/TEST OVERLAP (Filename Method)")
print("="*80)

def list_files(root):
    """Get set of all image filenames in directory."""
    return set([p.name for p in Path(root).rglob("*.jpeg")])

# Get all filenames
train_files = list_files(TRAIN_PATH)
test_files = list_files(TEST_PATH)

print(f"\nTrain files: {len(train_files):,}")
print(f"Test files: {len(test_files):,}")

# Check overlap
overlap = train_files.intersection(test_files)
print(f"\nFilename overlap: {len(overlap)}")

if len(overlap) > 0:
    print("❌ WARNING: Found overlapping files!")
    print("Examples:", list(overlap)[:10])
    raise ValueError("Train/test overlap detected - dataset not clean!")
else:
    print("✅ No filename overlap detected")
    print("   Dataset is clean - safe to proceed with training")

print("="*80)

VERIFYING NO TRAIN/TEST OVERLAP (Filename Method)

Train files: 55,792
Test files: 968

Filename overlap: 0
✅ No filename overlap detected
   Dataset is clean - safe to proceed with training


In [6]:
# SE-RESNET50 MODEL DEFINITION

# MODEL INITIALIZATION - SE-RESNET50


# SE Block Definition
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# SE-ResNet50 Model
class SEResNet50(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, reduction=16):
        super(SEResNet50, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        
        # Copy backbone layers
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # Add SE blocks after each layer
        self.se1 = SEBlock(256, reduction)
        self.se2 = SEBlock(512, reduction)
        self.se3 = SEBlock(1024, reduction)
        self.se4 = SEBlock(2048, reduction)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.se1(x)  # SE after layer1
        
        x = self.layer2(x)
        x = self.se2(x)  # SE after layer2
        
        x = self.layer3(x)
        x = self.se3(x)  # SE after layer3
        
        x = self.layer4(x)
        x = self.se4(x)  # SE after layer4
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


# Create model
model = SEResNet50(num_classes=NUM_CLASSES, pretrained=True, reduction=16)
model = model.to(DEVICE)

# Loss (with class weights for imbalance)
class_weights = torch.tensor([
    len(train_labels) / (NUM_CLASSES * train_counts[i])
    for i in range(NUM_CLASSES)
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("SE-ResNet50 initialized")
print(f"Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"SE reduction ratio: 16")
print(f"Class weights: {class_weights.cpu().numpy()}")

SE-ResNet50 initialized
Parameters: ~24.2M
SE reduction ratio: 16
Class weights: [0.62378985 2.0224752  3.614558   0.6150843 ]


In [7]:
# MODEL INITIALIZATION

# Create model
model = SEResNet50(num_classes=NUM_CLASSES, pretrained=True, reduction=SE_REDUCTION)
model = model.to(DEVICE)

# Loss (with class weights for imbalance)
class_weights = torch.tensor([
    len(train_labels) / (NUM_CLASSES * train_counts[i])
    for i in range(NUM_CLASSES)
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("Model initialized")
print(f"SE reduction ratio: {SE_REDUCTION}")
print(f"Class weights: {class_weights.cpu().numpy()}")

Model initialized
SE reduction ratio: 16
Class weights: [0.62378985 2.0224752  3.614558   0.6150843 ]


In [8]:
# IMPROVED TRAINING LOOP WITH MULTI-METRIC MONITORING

print("\n" + "="*80)
print(f"STARTING TRAINING - {MODEL_NAME.upper()}")
print("="*80)
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Val size: {len(val_dataset):,} images (much better than 32!)")
print("="*80)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_f1': [], 'val_precision': [], 'val_recall': [],
    'composite_score': [],
    'learning_rates': [],
    'overfitting_checks': []
}

# Robust initialization
best_composite_score = float('-inf')  # Handles negative scores
best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = -1  # -1 indicates "not set yet"

start_time = time.time()

try:
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        print("-" * 70)
        
        # === TRAINING PHASE ===
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        train_loss = train_loss / len(train_dataset)
        train_acc = 100.0 * train_correct / train_total
        
        # === VALIDATION PHASE WITH METRICS ===
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss / len(val_dataset)
        val_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))

        # Scale assertions
        assert 0 <= train_acc <= 100, f"Train acc {train_acc:.2f} not in [0,100]"
        assert 0 <= val_acc <= 100, f"Val acc {val_acc:.2f} not in [0,100]"
        
        # Calculate additional metrics
        val_f1 = f1_score(all_labels, all_preds, average='macro') * 100
        val_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        val_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        
        # Composite score (weighted combination)
        composite_score = (
            0.40 * val_acc +
            0.25 * val_f1 +
            0.20 * (100 - min(val_loss * 10, 100)) +
            0.15 * max(0, 100 - abs(train_acc - val_acc) * 2)
        )
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['composite_score'].append(composite_score)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        scheduler.step(val_loss)
        
        # === BEST MODEL SELECTION ===
        # Deterministic tie-breaking
        is_best = is_better_model(
            new_score=composite_score,
            new_loss=val_loss,
            new_acc=val_acc,
            new_epoch=epoch + 1,
            best_score=best_composite_score,
            best_loss=best_val_loss,
            best_acc=best_val_acc,
            best_epoch=best_epoch
        )
        
        if is_best:
            best_composite_score = composite_score
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, epoch + 1, metrics, True,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'best')
        
        # Periodic checkpoints
        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, epoch + 1, metrics, False,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'intermediate')
        
        # === OVERFITTING CHECK ===
        if (epoch + 1) % OVERFITTING_CHECK_INTERVAL == 0:
            overfit_check = check_overfitting(train_acc, val_acc, train_loss, val_loss)
            history['overfitting_checks'].append((epoch + 1, overfit_check))
            
            if overfit_check['is_overfitting']:
                print(f"\n⚠️ OVERFITTING WARNING [{overfit_check['severity']}]:")
                print(f"   Train-Val Acc Gap: {overfit_check['acc_gap']:.2f}%")
                print(f"   Val-Train Loss Gap: {overfit_check['loss_gap']:.4f}")
                print(f"   Consider: Early stopping, more regularization, or data augmentation")
        
        # === EPOCH SUMMARY ===
        epoch_time = time.time() - epoch_start
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
        print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
        print(f"  Val:   F1={val_f1:.2f}%, Prec={val_precision:.2f}%, Rec={val_recall:.2f}%")
        print(f"  Composite Score: {composite_score:.2f}")
        if is_best:
            print(f"  🎯 NEW BEST MODEL!")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {epoch_time:.1f}s")
        print("=" * 70)

except KeyboardInterrupt:
    print("\n\n⚠️ TRAINING INTERRUPTED BY USER")
    print(f"Completed {epoch + 1}/{NUM_EPOCHS} epochs")
    print(f"Best model saved at epoch {best_epoch}")

# SAVE FINAL CHECKPOINT
final_metrics = {
    'train_loss': train_loss, 'train_acc': train_acc,
    'val_loss': val_loss, 'val_acc': val_acc,
    'val_f1': val_f1, 'composite_score': composite_score
}

save_checkpoint(model, optimizer, epoch + 1, final_metrics, False,
              CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'last')

# TRAINING COMPLETE
total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*80)
print("** TRAINING COMPLETE **")
print("="*80)
print(f"Best model (by composite score): Epoch {best_epoch}")
print(f"  Composite Score: {best_composite_score:.2f}")
print(f"  Val Accuracy: {best_val_acc:.2f}%")
print(f"  Val Loss: {best_val_loss:.4f}")
print(f"\nTotal training time: {hours}h {minutes}m")
print(f"Serial number: {SERIAL_NUMBER:02d}")
print(f"Checkpoints saved: {CHECKPOINT_DIR}")
print("="*80)

# Save training history
history_file = CHECKPOINT_DIR / f"{SERIAL_NUMBER:02d}_{MODEL_NAME}_seed{SEED}_history.json"
with open(history_file, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    history_serializable = {k: [float(x) if isinstance(x, (np.floating, np.integer)) else x 
                                for x in v] if isinstance(v, list) else v 
                           for k, v in history.items()}
    json.dump(history_serializable, f, indent=2)

print(f"\nTraining history saved: {history_file.name}")
print("\n✓ Use Master_Evaluation.ipynb for final test set evaluation")
print("="*80)


STARTING TRAINING - SE_RESNET
Serial: 05 | Seed: 84 | Epochs: 50
Device: cuda
Val size: 8,369 images (much better than 32!)

Epoch [1/50]
----------------------------------------------------------------------


Saved best: 05_se_resnet_seed84_epoch1_best_20260114_122059.pth

Epoch 1 Summary:
  Train: Loss=0.5016, Acc=84.72%
  Val:   Loss=0.3735, Acc=91.62%
  Val:   F1=86.47%, Prec=85.66%, Rec=87.39%
  Composite Score: 90.45
  🎯 NEW BEST MODEL!
  LR: 0.001000 | Time: 122.2s

Epoch [2/50]
----------------------------------------------------------------------



Epoch 2 Summary:
  Train: Loss=0.3503, Acc=89.77%
  Val:   Loss=0.4388, Acc=86.69%
  Val:   F1=80.35%, Prec=78.82%, Rec=85.59%
  Composite Score: 87.96
  LR: 0.001000 | Time: 121.5s

Epoch [3/50]
----------------------------------------------------------------------


Saved best: 05_se_resnet_seed84_epoch3_best_20260114_122502.pth

Epoch 3 Summary:
  Train: Loss=0.3242, Acc=90.11%
  Val:   Loss=0.3310, Acc=90.86%
  Val:   F1=85.80%, Prec=84.57%, Rec=88.10%
  Composite Score: 91.91
  🎯 NEW BEST MODEL!
  LR: 0.001000 | Time: 122.1s

Epoch [4/50]
----------------------------------------------------------------------


Saved best: 05_se_resnet_seed84_epoch4_best_20260114_122704.pth

Epoch 4 Summary:
  Train: Loss=0.2933, Acc=91.30%
  Val:   Loss=0.2723, Acc=91.22%
  Val:   F1=86.63%, Prec=84.98%, Rec=90.62%
  Composite Score: 92.57
  🎯 NEW BEST MODEL!
  LR: 0.001000 | Time: 121.4s

Epoch [5/50]
----------------------------------------------------------------------


...


Epoch 46 Summary:
  Train: Loss=0.0896, Acc=96.97%
  Val:   Loss=0.1440, Acc=96.38%
  Val:   F1=93.85%, Prec=92.39%, Rec=95.55%
  Composite Score: 96.55
  LR: 0.000125 | Time: 120.5s

Epoch [47/50]
----------------------------------------------------------------------



Epoch 47 Summary:
  Train: Loss=0.0878, Acc=96.98%
  Val:   Loss=0.1425, Acc=96.33%
  Val:   F1=93.72%, Prec=92.27%, Rec=95.42%
  Composite Score: 96.48
  LR: 0.000125 | Time: 120.7s

Epoch [48/50]
----------------------------------------------------------------------



Epoch 48 Summary:
  Train: Loss=0.0886, Acc=96.93%
  Val:   Loss=0.1327, Acc=96.14%
  Val:   F1=93.41%, Prec=91.68%, Rec=95.53%
  Composite Score: 96.31
  LR: 0.000125 | Time: 120.5s

Epoch [49/50]
----------------------------------------------------------------------



Epoch 49 Summary:
  Train: Loss=0.0882, Acc=97.03%
  Val:   Loss=0.1468, Acc=96.42%
  Val:   F1=93.77%, Prec=92.45%, Rec=95.36%
  Composite Score: 96.53
  LR: 0.000125 | Time: 120.5s

Epoch [50/50]
----------------------------------------------------------------------


Saved intermediate: 05_se_resnet_seed84_epoch50_intermediate_20260114_135952.pth

Epoch 50 Summary:
  Train: Loss=0.0874, Acc=96.98%
  Val:   Loss=0.1544, Acc=96.10%
  Val:   F1=93.31%, Prec=92.00%, Rec=94.79%
  Composite Score: 96.20
  LR: 0.000125 | Time: 120.7s
Saved last: 05_se_resnet_seed84_epoch50_last_20260114_135952.pth

** TRAINING COMPLETE **
Best model (by composite score): Epoch 42
  Composite Score: 96.84
  Val Accuracy: 96.62%
  Val Loss: 0.1383

Total training time: 1h 40m
Serial number: 05
Checkpoints saved: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints

Training history saved: 05_se_resnet_seed84_history.json

✓ Use Master_Evaluation.ipynb for final test set evaluation
